In [ ]:
%run "../scripts/load_daily_variables.py"

Loading Variables: 100%|██████████| 26/26 [07:34<00:00, 17.49s/it]  


: 

# Needs
- List of ETFs that can be the benchmark
- Access to lists of symbols that can be compared to the benchmark
- Start date input
## Symbol list source
- Local files
- Finviz ETF membership function
- Sector industry member search function from fundamentals.py

# Future add-ons
- Plot for relative strength
- Quant rating
- Intraday relative strength
- Intraday correlation/time-lagged cross correlation

In [ ]:
from api_keys import finviz_api_key
from io import StringIO

def etf_membership(etf):
    url = f"https://elite.finviz.com/export/screener?v=111&f=etf_heldby_{etf}&ft=5&auth={finviz_api_key}"
    response = requests.get(url)
    result = pd.read_csv(StringIO(response.text))
    return result

etf_membership('IGV')

In [ ]:
igv = [symbol for symbol in etf_membership('IGV').Ticker.to_list() if symbol in symbols]
software = (
    fu.sector_industry_member_search('Software - Infrastructure', level='industry') +
    fu.sector_industry_member_search('Software - Application', level='industry')
)
smh = [symbol for symbol in etf_membership('SMH').Ticker.to_list() if symbol in symbols]
semiconductors = fu.sector_industry_member_search('Semiconductors', level='industry')
xme = [symbol for symbol in etf_membership('XME').Ticker.to_list() if symbol in symbols]

In [ ]:
from support_functions import relative

df = pd.DataFrame()

for sym in semiconductors:
    if (sym in symbols) and ('2026-06-22' in symbols[sym].df.index):
        s = relative(
            df=symbols[sym].df,
            bm=etfs['SMH'].df['Close'],
            start='2026-06-22',
        ).Relative_Close
        s.name = sym
        df = pd.concat([df, s], axis=1)

In [ ]:
df.iloc[-1].describe()

- From .describe() use ceiling rounded value of 25th perentile for lower range and floor rounded value of 75th percentile for upper range.

In [ ]:
df.loc[:, (df.iloc[-1] < 80) | (df.iloc[-1] > 97)].iloc[-1].sort_values(ascending=False)

- After core functionality is established add-ons can be included.
- Can plot the relative strength value for visual observation.
- Can use the drawdown function to find symbols at extreme relative strength/weakness compared to their prior values.
- It may also be worth identifying symbols with increasing/decreasing relative strength/weakness.

In [ ]:
df.STM.plot()

In [ ]:
from stats_objects import drawdown_current,drawdown_max